# CuVision on Colab

A CUDA image-processing engine where every operation has a single-threaded CPU
reference and two or three GPU kernels that compute the *same* result by
different means. The benchmark prints throughput next to the maximum per-pixel
error against the CPU oracle, so a "faster" kernel that quietly changes its
output cannot hide.

**Before you run anything:** Runtime -> Change runtime type -> Hardware
accelerator -> **GPU** (a T4 is plenty).

Repo: https://github.com/Snehasis95/CuVision

## 1. Check the GPU

If `nvidia-smi` fails, the runtime has no accelerator attached.

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Clone and build

The Makefile defaults to `-arch=native`, which needs CUDA >= 11.5. Colab's
toolkit is newer than that, so it detects the attached card on its own. On an
older toolkit, build with `make ARCH=-arch=sm_75` instead.

In [ ]:
%cd /content
!rm -rf CuVision
!git clone --depth 1 https://github.com/Snehasis95/CuVision.git
%cd /content/CuVision
!make -j$(nproc)

## 3. Host tests

CUDA-free: netpbm round-tripping, Gaussian weight normalisation and symmetry,
blur preserving a constant image, Sobel firing on a step edge but not on flat
regions, histogram totals, LUT monotonicity, and rejection of bad input.

These are the same reference implementations the GPU kernels are checked
against, so they run first.

In [ ]:
!make test

## 4. What card did we get?

In [ ]:
!./imgproc info

## 5. Run the operations

`gen` writes a synthetic RGB test pattern, so nothing needs downloading.
`pipeline` chains grayscale -> blur -> Sobel with one upload and one download;
the intermediates never leave the device.

In [ ]:
!mkdir -p data
!./imgproc gen      data/test.ppm 1920 1080
!./imgproc gray     data/test.ppm data/gray.pgm
!./imgproc blur     data/test.ppm data/blur.pgm 8
!./imgproc sobel    data/test.ppm data/sobel.pgm
!./imgproc equalize data/test.ppm data/equalized.pgm
!./imgproc pipeline data/test.ppm data/edges.pgm 5

### Look at the results

Outputs are binary netpbm (P6 for RGB, P5 for grayscale), which PIL reads directly.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

shots = [
    ("data/test.ppm",      "source (synthetic RGB)"),
    ("data/gray.pgm",      "grayscale"),
    ("data/blur.pgm",      "gaussian blur r=8"),
    ("data/sobel.pgm",     "sobel magnitude"),
    ("data/equalized.pgm", "histogram equalised"),
    ("data/edges.pgm",     "pipeline: gray -> blur -> sobel"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (path, title) in zip(axes.ravel(), shots):
    img = Image.open(path)
    ax.imshow(img, cmap=None if img.mode == "RGB" else "gray", vmin=None if img.mode == "RGB" else 0,
              vmax=None if img.mode == "RGB" else 255)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Benchmark

Device buffers are allocated once and the kernels are warmed up, then `iters`
launches are averaged inside a single `cudaEvent` pair — so the timings exclude
`cudaMalloc` and host launch overhead. PCIe transfer cost is measured and
printed separately.

4096x4096 at radius 8 takes a minute or so, mostly in the single-threaded CPU
reference for the naive blur.

In [ ]:
!./imgproc bench 4096 4096 8 50

### Reading the output

**`max err`** is the largest absolute per-byte difference against the CPU
implementation. 0-1 is float contraction, not a bug. The end-to-end figure is
larger by design: it chains the *GPU* grayscale, and Sobel's weights (summing
to 8) amplify a 1-LSB input difference. That is exactly why the per-kernel
comparisons feed the CPU's grayscale to the later stages instead.

**What to look for:**

* **grayscale** — `vec4` reads 4 px as three 32-bit loads instead of a 3-byte
  stride. Compare its effective GB/s against the peak bandwidth printed by
  `info`; this kernel moves 4 bytes per pixel and should get close.
* **blur** — separable drops per-pixel work from `(2r+1)^2` to `2*(2r+1)`:
  289 multiply-adds against 34 at r=8. That, not tiling, is the real win.
* **sobel** — tiling usually *loses* here. A 3x3 stencil is small enough that L1
  absorbs the redundant reads, so the tile only buys a `__syncthreads()` and a
  halo load. On a T4 at 16.8 MPixel the naive kernel measured 0.527 ms against
  0.783 ms tiled, which is why `run_sobel` and `run_pipeline` use it.
* **histogram** — privatising the 256 bins per block keeps atomic contention in
  shared memory. Most pixels land in a few hot bins, so the gap is wide.
* **transfers** — for a single pass over one image, PCIe dominates every kernel
  in the table. That is the argument for the fused `pipeline` command.

## 7. Try your own image

CuVision reads binary netpbm only. Convert anything else first — PIL will do
it in one line.

In [ ]:
# from PIL import Image
# Image.open("/content/your_photo.jpg").convert("RGB").save("data/mine.ppm")
# !./imgproc pipeline data/mine.ppm data/mine_edges.pgm 5
# Image.open("data/mine_edges.pgm")

## 8. Profiling

`-lineinfo` is compiled in, so Nsight Compute maps stalls back to source lines.
Colab allows the CLI profiler; the `--set full` metric groups need permissions
Colab does not grant, so stick to a named section.

In [ ]:
# !ncu --set detailed --section MemoryWorkloadAnalysis #      --kernel-name regex:k_blur --launch-count 1 #      ./imgproc bench 2048 2048 8 5